## 0) Setup

In [ ]:
import os, random, time, math
from pathlib import Path
from sklearn.model_selection import StratifiedShuffleSplit

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
import torchvision
from torchvision import transforms
from PIL import Image

from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, Optional

import torch
import torch.nn as nn

import csv
from glob import glob

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())


===> Starting: imports loaded
PyTorch: 2.10.0+cu130
Torchvision: 0.25.0+cu130
CUDA available: True


## 1) Configuration

In [ ]:
# ===== User config =====
DATA_ROOT = Path("C:\\Users\\leino\\Documents\\JYU Opinnot\\TIES4700 - Deep Learning\\RODI-DATA")
TRAIN_DIR = DATA_ROOT / "Train_iso"
TEST_DIR  = DATA_ROOT / "Test"   # images directly inside this folder

MODEL_NAME = "resnet18"  # "resnet18" or "cnn"
IMG_SIZE = 224

BATCH_SIZE = 64
NUM_EPOCHS = 20
LR = 3e-4
WEIGHT_DECAY = 1e-4

NUM_WORKERS = 0
SEED = 42

# Early stopping metric:
# choose one: "auroc" or "auprc" or "accuracy"
BEST_METRIC_NAME = "auroc"

# Where to save outputs
OUT_DIR = Path("outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

CKPT_PATH = OUT_DIR / f"best_{MODEL_NAME}.pth"
PRED_CSV_PATH = OUT_DIR / "predictions.csv"

print("TRAIN_DIR:", TRAIN_DIR.resolve())
print("TEST_DIR :", TEST_DIR.resolve())
print("MODEL_NAME:", MODEL_NAME)


===> Starting: configuration set
TRAIN_DIR: C:\Users\leino\Documents\JYU Opinnot\TIES4700 - Deep Learning\RODI-DATA\Train_iso
TEST_DIR : C:\Users\leino\Documents\JYU Opinnot\TIES4700 - Deep Learning\RODI-DATA\Test
MODEL_NAME: resnet18


## 2) Reproducibility + device

In [ ]:
def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # deterministic can be slower; enable if you need exact reproducibility
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

seed_everything(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(" device selected ->", device)


===> Starting: device selected -> cuda


## 3) Transforms: pad-to-square (keep aspect ratio) → resize 224×224 → tensor → normalize

In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

class PadToSquare:
    """
    Pads an image with zeros (black) to make it square, keeping aspect ratio.
    Then downstream Resize can make it 224x224.
    """
    def __init__(self, fill=0):
        self.fill = fill

    def __call__(self, img: Image.Image) -> Image.Image:
        if img.mode != "RGB":
            img = img.convert("RGB")
        w, h = img.size
        if w == h:
            return img
        size = max(w, h)
        new_img = Image.new("RGB", (size, size), (self.fill, self.fill, self.fill))
        left = (size - w) // 2
        top = (size - h) // 2
        new_img.paste(img, (left, top))
        return new_img

train_tfms = transforms.Compose([
    PadToSquare(fill=0),
    transforms.Resize((IMG_SIZE, IMG_SIZE), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_tfms = transforms.Compose([
    PadToSquare(fill=0),
    transforms.Resize((IMG_SIZE, IMG_SIZE), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

===> Starting: transforms ready


## 4) Dataset + stratified 80/20 train/val split

In [ ]:
if not TRAIN_DIR.exists():
    raise FileNotFoundError(f"TRAIN_DIR not found: {TRAIN_DIR.resolve()}")

full_ds = torchvision.datasets.ImageFolder(root=str(TRAIN_DIR), transform=val_tfms)
class_to_idx = full_ds.class_to_idx
print("Classes found:", class_to_idx)

# We want fish=1 and non-fish=0.
# ImageFolder assigns indices alphabetically. Usually artifacts=0, fish=1 (good).
# If not, we remap labels inside our subsets.
if "fish" not in class_to_idx:
    raise ValueError(f"Could not find a 'fish' folder under {TRAIN_DIR}. Found: {list(class_to_idx.keys())}")

fish_idx = class_to_idx["fish"]

targets = np.array(full_ds.targets)  # assigned by ImageFolder
# Map to binary labels: fish=1 else 0
binary_targets = (targets == fish_idx).astype(np.int64)

# ---- Stratified 80/20 split ----
def stratified_split_indices(y: np.ndarray, train_frac: float = 0.8, seed: int = 42):
    # Prefer sklearn if available
    try:
        sss = StratifiedShuffleSplit(n_splits=1, test_size=1-train_frac, random_state=seed)
        train_idx, val_idx = next(sss.split(np.zeros(len(y)), y))
        return train_idx.tolist(), val_idx.tolist()
    except Exception as e:
        # Fallback: manual stratification
        rng = np.random.default_rng(seed)
        idx0 = np.where(y == 0)[0]
        idx1 = np.where(y == 1)[0]
        rng.shuffle(idx0); rng.shuffle(idx1)
        n0_train = int(len(idx0) * train_frac)
        n1_train = int(len(idx1) * train_frac)
        train_idx = np.concatenate([idx0[:n0_train], idx1[:n1_train]])
        val_idx = np.concatenate([idx0[n0_train:], idx1[n1_train:]])
        rng.shuffle(train_idx); rng.shuffle(val_idx)
        return train_idx.tolist(), val_idx.tolist()

train_idx, val_idx = stratified_split_indices(binary_targets, train_frac=0.8, seed=SEED)

# Create separate datasets so they can use different transforms (augmentation only on train)
train_ds = torchvision.datasets.ImageFolder(root=str(TRAIN_DIR), transform=train_tfms)
val_ds   = torchvision.datasets.ImageFolder(root=str(TRAIN_DIR), transform=val_tfms)

# Wrap with Subset
train_subset = Subset(train_ds, train_idx)
val_subset   = Subset(val_ds, val_idx)

# Store binary labels for subsets (useful for metrics sanity checks)
train_y = binary_targets[train_idx]
val_y   = binary_targets[val_idx]

print(f"Train size: {len(train_subset)}  (fish: {int(train_y.sum())}, non-fish: {int((train_y==0).sum())})")
print(f"Val size  : {len(val_subset)}    (fish: {int(val_y.sum())}, non-fish: {int((val_y==0).sum())})")


===> Starting: dataset scanned
Classes found: {'artifact': 0, 'fish': 1, 'insect': 2, 'leave': 3, 'partial-overlapping-objects': 4, 'rounded': 5, 'shadow': 6, 'variable': 7, 'wormish': 8}
===> Starting: stratified split created
Train size: 38276  (fish: 214, non-fish: 38062)
Val size  : 9569    (fish: 54, non-fish: 9515)


## 5) DataLoaders

In [ ]:
train_loader = DataLoader(
    train_subset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available()
)
val_loader = DataLoader(
    val_subset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available()
)

===> Starting: dataloaders ready
len(train_subset) = 38276
len(val_subset)   = 9569
len(train_loader) = 1197
len(val_loader)   = 300


## 6) Models

In [ ]:
class SimpleCNN(nn.Module):
    """A compact CNN for 224x224 RGB images. Outputs a single logit."""
    def __init__(self, dropout: float = 0.3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),

            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
        )
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(p=dropout),
            nn.Linear(256, 1),
        )

        self._init_weights()

    def _init_weights(self):
        # Kaiming init for conv layers, zero bias.
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, mean=0.0, std=0.01)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = self.pool(x)
        x = self.classifier(x)
        return x  # [B, 1] logits


class ResNet18Binary(nn.Module):
    """ResNet18 backbone with 1-unit FC head. Uses random weights unless specified otherwise."""
    def __init__(self, weights=None):
        super().__init__()
        self.backbone = torchvision.models.resnet18(weights=weights)
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Linear(in_features, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.backbone(x)  # [B, 1] logits


def create_model(model_name: str) -> nn.Module:
    """
    model_name:
      - "resnet18"
      - "cnn"
    """
    name = model_name.lower().strip()
    if name in {"resnet18", "resnet-18"}:
        # IMPORTANT: random weights -> weights=None
        return ResNet18Binary(weights=None)
    if name in {"cnn", "simplecnn", "simple_cnn"}:
        return SimpleCNN()
    raise ValueError(f"Unknown model_name={model_name!r}. Use 'resnet18' or 'cnn'.")


def load_checkpoint(ckpt_path: str, model: nn.Module, device: torch.device | str = "cpu") -> dict:
    """Load a checkpoint into a model and return the full checkpoint dict."""
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    return ckpt



## 7) Metrics (Precision/Recall per class, Accuracy, AUPRC, AUROC)

In [ ]:

def confusion_from_probs(y_true: np.ndarray, y_prob: np.ndarray, threshold: float = 0.5):
    """
    y_true: {0,1}
    y_prob: [0,1]
    """
    y_pred = (y_prob >= threshold).astype(np.int64)
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    return tp, tn, fp, fn

def safe_div(a, b):
    return float(a) / float(b) if b != 0 else 0.0

def metrics_from_probs(y_true: np.ndarray, y_prob: np.ndarray, threshold: float = 0.5):
    tp, tn, fp, fn = confusion_from_probs(y_true, y_prob, threshold=threshold)

    # Per-class precision/recall
    # fish is positive class (1)
    prec_fish = safe_div(tp, tp + fp)
    rec_fish  = safe_div(tp, tp + fn)

    # non-fish is class 0; treat it as "positive" for its own precision/recall:
    # predict non-fish when y_pred==0
    prec_nonfish = safe_div(tn, tn + fn)
    rec_nonfish  = safe_div(tn, tn + fp)

    acc = safe_div(tp + tn, tp + tn + fp + fn)

    # AUROC & AUPRC (prefer sklearn if available)
    auroc = None
    auprc = None
    try:
        from sklearn.metrics import roc_auc_score, average_precision_score
        # Need both classes present; otherwise these can error
        if len(np.unique(y_true)) == 2:
            auroc = float(roc_auc_score(y_true, y_prob))
            auprc = float(average_precision_score(y_true, y_prob))
        else:
            auroc = float("nan")
            auprc = float("nan")
    except Exception:
        # Fallback: nan
        auroc = float("nan")
        auprc = float("nan")

    return {
        "precision_fish": prec_fish,
        "recall_fish": rec_fish,
        "precision_nonfish": prec_nonfish,
        "recall_nonfish": rec_nonfish,
        "accuracy": acc,
        "auroc": auroc,
        "auprc": auprc,
        "tp": tp, "tn": tn, "fp": fp, "fn": fn,
    }

===> Starting: metrics helpers ready


## 8) Training + validation loops

In [ ]:

def get_binary_label_from_imagefolder_label(lbl: torch.Tensor, fish_class_index: int) -> torch.Tensor:
    # lbl: ImageFolder multiclass label. Convert to binary fish/non-fish.
    return (lbl == fish_class_index).long()

@torch.no_grad()
def run_eval(model: nn.Module, loader: DataLoader, fish_class_index: int, device: torch.device):
    model.eval()
    probs_all = []
    y_all = []
    loss_all = []

    criterion = nn.BCEWithLogitsLoss()

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        y = get_binary_label_from_imagefolder_label(labels, fish_class_index).float().to(device)

        logits = model(images).squeeze(1)  # [B]
        loss = criterion(logits, y)
        prob = torch.sigmoid(logits)

        probs_all.append(prob.detach().cpu().numpy())
        y_all.append(y.detach().cpu().numpy())
        loss_all.append(loss.item())

    y_true = np.concatenate(y_all) if y_all else np.array([])
    y_prob = np.concatenate(probs_all) if probs_all else np.array([])
    m = metrics_from_probs(y_true.astype(np.int64), y_prob.astype(np.float64), threshold=0.5)
    m["loss"] = float(np.mean(loss_all)) if loss_all else float("nan")
    return m, y_true, y_prob

def run_train_one_epoch(model: nn.Module, loader: DataLoader, optimizer: torch.optim.Optimizer,
                        fish_class_index: int, device: torch.device, epoch: int):
    model.train()
    criterion = nn.BCEWithLogitsLoss()
    running_loss = 0.0
    n = 0

    for step, (images, labels) in enumerate(loader, start=1):
        images = images.to(device, non_blocking=True)
        y = get_binary_label_from_imagefolder_label(labels, fish_class_index).float().to(device)
    
        optimizer.zero_grad(set_to_none=True)
        logits = model(images).squeeze(1)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        bs = images.size(0)
        running_loss += loss.item() * bs
        n += bs

        if step == 1 or step % 50 == 0:
            print(f"    [epoch {epoch:03d}] step {step:04d} | loss {loss.item():.4f}")

    return running_loss / max(n, 1)

===> Starting: train/eval loops ready


## 9) Create model (random weights) + optimizer

In [ ]:

model = create_model(MODEL_NAME).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

print(model.__class__.__name__)
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))
print("Trying to fetch one batch...")
batch = next(iter(train_loader))
images, labels = batch
print("Batch OK:", images.shape, labels.shape, labels[:10])


===> Starting: model created with random weights
ResNet18Binary
Trainable parameters: 11177025
Trying to fetch one batch...
Batch OK: torch.Size([32, 3, 224, 224]) torch.Size([32]) tensor([8, 8, 8, 8, 3, 7, 3, 8, 7, 8])


## 10) Train with best-checkpoint saving

In [ ]:

def select_best_metric(metrics: dict, metric_name: str) -> float:
    if metric_name not in metrics:
        raise KeyError(f"Metric {metric_name!r} not found in: {list(metrics.keys())}")
    val = metrics[metric_name]
    # Handle None / nan
    if val is None:
        return float("-inf")
    try:
        if math.isnan(val):
            return float("-inf")
    except Exception:
        pass
    return float(val)

best_metric = float("-inf")
best_epoch = -1

print("===> Starting: training loop")
for epoch in range(1, NUM_EPOCHS + 1):
    print(f"\n====> Epoch {epoch}/{NUM_EPOCHS} (MODEL={MODEL_NAME})")

    t0 = time.time()
    train_loss = run_train_one_epoch(model, train_loader, optimizer, fish_idx, device, epoch)
    print(f"===> Train done | loss={train_loss:.4f} | time={time.time()-t0:.1f}s")

    print("===> Starting: validation")
    val_metrics, y_true_val, y_prob_val = run_eval(model, val_loader, fish_idx, device)

    # Pretty print required metrics
    print(f"Val loss: {val_metrics['loss']:.4f}")
    print(f"Accuracy: {val_metrics['accuracy']:.4f}")
    print(f"Precision (fish=1): {val_metrics['precision_fish']:.4f}")
    print(f"Recall    (fish=1): {val_metrics['recall_fish']:.4f}")
    print(f"Precision (non-fish=0): {val_metrics['precision_nonfish']:.4f}")
    print(f"Recall    (non-fish=0): {val_metrics['recall_nonfish']:.4f}")
    print(f"AUROC: {val_metrics['auroc']}")
    print(f"AUPRC: {val_metrics['auprc']}")
    print(f"Confusion (tp, tn, fp, fn): ({val_metrics['tp']}, {val_metrics['tn']}, {val_metrics['fp']}, {val_metrics['fn']})")

    metric_value = select_best_metric(val_metrics, BEST_METRIC_NAME)

    if metric_value > best_metric:
        best_metric = metric_value
        best_epoch = epoch

        ckpt = {
            "epoch": epoch,
            "best_metric": float(best_metric),
            "metric_name": BEST_METRIC_NAME,
            "model_name": MODEL_NAME,
            "class_to_idx": class_to_idx,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
        }
        torch.save(ckpt, CKPT_PATH)
        print(f"===> New best! Saved checkpoint -> {CKPT_PATH} (best {BEST_METRIC_NAME}={best_metric:.6f})")

print("\n===> Training finished")
print("Best epoch:", best_epoch, "| best", BEST_METRIC_NAME, "=", best_metric)


===> Starting: training loop

====> Epoch 1/10 (MODEL=resnet18)
    [epoch 001] step 0001 | loss 0.7298
    [epoch 001] step 0050 | loss 0.1824
    [epoch 001] step 0100 | loss 0.0030
    [epoch 001] step 0150 | loss 0.0035
    [epoch 001] step 0200 | loss 0.0064
    [epoch 001] step 0250 | loss 0.0055
    [epoch 001] step 0300 | loss 0.0038
    [epoch 001] step 0350 | loss 0.0065
    [epoch 001] step 0400 | loss 0.0046
    [epoch 001] step 0450 | loss 0.1420
    [epoch 001] step 0500 | loss 0.1520
    [epoch 001] step 0550 | loss 0.0100
    [epoch 001] step 0600 | loss 0.0072
    [epoch 001] step 0650 | loss 0.0062
    [epoch 001] step 0700 | loss 0.0031
    [epoch 001] step 0750 | loss 0.0042
    [epoch 001] step 0800 | loss 0.1614
    [epoch 001] step 0850 | loss 0.0101
    [epoch 001] step 0900 | loss 0.1611
    [epoch 001] step 0950 | loss 0.0074
    [epoch 001] step 1000 | loss 0.0025
    [epoch 001] step 1050 | loss 0.0141
    [epoch 001] step 1100 | loss 0.0034
    [epoch 001] 

## 11) Inference on test images → `predictions.csv`

In [ ]:
# Load best checkpoint (recommended)
if not CKPT_PATH.exists():
    raise FileNotFoundError(f"Checkpoint not found: {CKPT_PATH.resolve()}")

print("===> Starting: loading best checkpoint")
ckpt = torch.load(CKPT_PATH, map_location=device)
model = create_model(ckpt["model_name"]).to(device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()
print("Loaded checkpoint from epoch:", ckpt["epoch"], "| best", ckpt["metric_name"], "=", ckpt["best_metric"])

# Collect test image paths
if not TEST_DIR.exists():
    raise FileNotFoundError(f"TEST_DIR not found: {TEST_DIR.resolve()}")

exts = ("*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG")
test_paths = []
for e in exts:
    test_paths.extend(glob(str(TEST_DIR / e)))

test_paths = sorted(test_paths)
print(f"===> Starting: predicting on {len(test_paths)} test images")

def load_and_preprocess(img_path: str):
    img = Image.open(img_path).convert("RGB")
    x = val_tfms(img)  # same as validation transform
    return x

rows = [("img_name", "probability")]

with torch.no_grad():
    for i, p in enumerate(test_paths, start=1):
        x = load_and_preprocess(p).unsqueeze(0).to(device)
        logit = model(x).squeeze()
        prob = torch.sigmoid(logit).item()
        img_name = Path(p).name
        rows.append((img_name, f"{prob:.10f}"))
        if i == 1 or i % 200 == 0:
            print(f"    predicted {i}/{len(test_paths)}")

# Write CSV
PRED_CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(PRED_CSV_PATH, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerows(rows)

print("===> Done: wrote", PRED_CSV_PATH.resolve())


===> Starting: loading best checkpoint
Loaded checkpoint from epoch: 10 | best auroc = 0.9882349506626963
===> Starting: predicting on 31898 test images
    predicted 1/31898
    predicted 200/31898
    predicted 400/31898
    predicted 600/31898
    predicted 800/31898
    predicted 1000/31898
    predicted 1200/31898
    predicted 1400/31898
    predicted 1600/31898
    predicted 1800/31898
    predicted 2000/31898
    predicted 2200/31898
    predicted 2400/31898
    predicted 2600/31898
    predicted 2800/31898
    predicted 3000/31898
    predicted 3200/31898
    predicted 3400/31898
    predicted 3600/31898
    predicted 3800/31898
    predicted 4000/31898
    predicted 4200/31898
    predicted 4400/31898
    predicted 4600/31898
    predicted 4800/31898
    predicted 5000/31898
    predicted 5200/31898
    predicted 5400/31898
    predicted 5600/31898
    predicted 5800/31898
    predicted 6000/31898
    predicted 6200/31898
    predicted 6400/31898
    predicted 6600/31898
    